In [1]:
from pathlib import Path
import json

INTERSECTIONS_DIR = Path(r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\intersections")

def list_json_files(folder: Path):
    return sorted([p for p in folder.glob("*.json") if p.is_file()])

def load_tlc_by_alias(intersections_dir: Path, alias_target: str):
    """
    Finds the intersection definition JSON where tlc.alias == alias_target (or tlc.name matches),
    returns tlc dict and the filepath.
    """
    for fp in list_json_files(intersections_dir):
        with open(fp, "r", encoding="utf-8") as f:
            data = json.load(f)
        tlc = data.get("tlc", {})
        if tlc.get("alias") == alias_target or tlc.get("name") == alias_target:
            return tlc, fp
    return None, None

def extract_detectors(tlc: dict):
    """
    Returns list of (id_int, name, detectorType) from softwareStatus/node/detectors
    """
    out = []
    for sw in tlc.get("softwareStatus", []):
        node = sw.get("node", {})
        for d in node.get("detectors", []):
            d_id = d.get("id")
            d_name = d.get("name")
            d_type = d.get("detectorType")
            if d_id is None or not d_name:
                continue
            out.append((int(d_id), d_name, d_type))
    # unique by id (just in case)
    uniq = {}
    for d_id, d_name, d_type in out:
        uniq[d_id] = (d_id, d_name, d_type)
    return [uniq[k] for k in sorted(uniq.keys())]

def extract_fault_detectors(tlc: dict):
    """
    Returns list of (id_int, name, detectorType) from hardwareStatus.faultDetectors
    """
    hw = tlc.get("hardwareStatus", {})
    faults = []
    for fd in hw.get("faultDetectors", []) or []:
        fd_id = fd.get("id")
        fd_name = fd.get("name")
        fd_type = fd.get("detectorType")
        if fd_id is None:
            continue
        faults.append((int(fd_id), fd_name, fd_type))
    # unique by id
    uniq = {}
    for d_id, d_name, d_type in faults:
        uniq[d_id] = (d_id, d_name, d_type)
    return [uniq[k] for k in sorted(uniq.keys())]

def print_controller_summary(tlc: dict, alias_target: str, fp: Path):
    print(f"\n==============================")
    print(f"STATE CHECK: {alias_target}")
    print(f"==============================")
    print("File:", fp)
    print("name:", tlc.get("name"))
    print("alias:", tlc.get("alias"))
    print("active:", tlc.get("active"))
    print("state:", tlc.get("state"))
    print("brokenDetectorCount:", tlc.get("brokenDetectorCount"))
    print("lastStatusUpdate:", tlc.get("lastStatusUpdate"))
    print("lastContacted:", tlc.get("lastContacted"))
    hw = tlc.get("hardwareStatus", {})
    print("powerOk:", hw.get("powerOk"), "| emergencyOff:", hw.get("emergencyOff"), "| doorOpen:", hw.get("doorOpen"), "| persistenceOk:", hw.get("persistenceOk"))

**LD-LSA16 Failure Check**

This cell loads LD-LSA16 from the intersection definition files and performs a controller-level failure check.
It prints the controller state and lists detectors that are reported as faulty by the controller (hardwareStatus.faultDetectors).
Detectors reported as faulty are marked as UNUSABLE at this stage.

In [2]:
alias_target = "LD-LSA16"

tlc, fp = load_tlc_by_alias(INTERSECTIONS_DIR, alias_target)
if tlc is None:
    raise FileNotFoundError(f"Could not find {alias_target} in {INTERSECTIONS_DIR}")

print_controller_summary(tlc, alias_target, fp)

all_detectors = extract_detectors(tlc)
fault_detectors = extract_fault_detectors(tlc)

fault_ids = {d_id for d_id, _, _ in fault_detectors}

print("\n--- Controller-reported faulty detectors (UNUSABLE) ---")
if not fault_detectors:
    print("(none)")
else:
    for d_id, d_name, d_type in fault_detectors:
        print(f"- ID={d_id} | Name={d_name} | Type={d_type}")

print("\n--- All detectors in intersection definition ---")
for d_id, d_name, d_type in all_detectors:
    tag = "❌ FAULT" if d_id in fault_ids else "✅ OK"
    print(f"{tag} | ID={d_id:>2} | Name={d_name:<12} | Type={d_type}")

usable = [(d_id, d_name) for d_id, d_name, _ in all_detectors if d_id not in fault_ids]
unusable = [(d_id, d_name) for d_id, d_name, _ in all_detectors if d_id in fault_ids]

print("\n=== USABLE (snapshot-level) ===")
for d_id, d_name in usable:
    print(f" ✅ ID={d_id} | {d_name}")

print("\n=== UNUSABLE (snapshot-level) ===")
if not unusable:
    print("(none)")
else:
    for d_id, d_name in unusable:
        print(f" ❌ ID={d_id} | {d_name}")


STATE CHECK: LD-LSA16
File: C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\intersections\LSA16.json
name: LD-LSA16
alias: LD-LSA16
active: True
state: DETECTOR_FAILURE
brokenDetectorCount: 2
lastStatusUpdate: 2025-06-04T09:30:37
lastContacted: 2025-06-04T09:45:02.427
powerOk: True | emergencyOff: False | doorOpen: False | persistenceOk: True

--- Controller-reported faulty detectors (UNUSABLE) ---
- ID=8 | Name=WD41 | Type=NONE
- ID=9 | Name=WD42 | Type=NONE

--- All detectors in intersection definition ---
✅ OK | ID= 1 | Name=VD11         | Type=NONE
✅ OK | ID= 2 | Name=VD12         | Type=NONE
✅ OK | ID= 3 | Name=WD21         | Type=NONE
✅ OK | ID= 4 | Name=WD22         | Type=NONE
✅ OK | ID= 5 | Name=VD23         | Type=NONE
✅ OK | ID= 6 | Name=VD31         | Type=NONE
✅ OK | ID= 7 | Name=VD32         | Type=NONE
❌ FAULT | ID= 8 | Name=WD41         | Type=NONE
❌ FAULT | ID= 9 | Name=WD42         | Type=NONE
✅ OK | ID=10 | Na

**LD-LSA10 Failure Check**

In [3]:
alias_target = "LD-LSA10"

tlc, fp = load_tlc_by_alias(INTERSECTIONS_DIR, alias_target)
if tlc is None:
    raise FileNotFoundError(f"Could not find {alias_target} in {INTERSECTIONS_DIR}")

print_controller_summary(tlc, alias_target, fp)

all_detectors = extract_detectors(tlc)
fault_detectors = extract_fault_detectors(tlc)
fault_ids = {d_id for d_id, _, _ in fault_detectors}

print("\n--- Controller-reported faulty detectors (UNUSABLE) ---")
if not fault_detectors:
    print("(none)")
else:
    for d_id, d_name, d_type in fault_detectors:
        print(f"- ID={d_id} | Name={d_name} | Type={d_type}")

print("\n--- All detectors in intersection definition ---")
for d_id, d_name, d_type in all_detectors:
    tag = "❌ FAULT" if d_id in fault_ids else "✅ OK"
    print(f"{tag} | ID={d_id:>2} | Name={d_name:<12} | Type={d_type}")

print("\n=== USABLE (snapshot-level) ===")
for d_id, d_name, _ in all_detectors:
    if d_id not in fault_ids:
        print(f" ✅ ID={d_id} | {d_name}")

print("\n=== UNUSABLE (snapshot-level) ===")
if not fault_detectors:
    print("(none)")
else:
    for d_id, d_name, _ in fault_detectors:
        print(f" ❌ ID={d_id} | {d_name}")


STATE CHECK: LD-LSA10
File: C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\intersections\LSA10.json
name: LD-LSA10
alias: LD-LSA10
active: True
state: NO_FAILURE
brokenDetectorCount: 0
lastStatusUpdate: 2025-06-04T09:30:00
lastContacted: 2025-06-04T09:44:36.75
powerOk: True | emergencyOff: False | doorOpen: False | persistenceOk: True

--- Controller-reported faulty detectors (UNUSABLE) ---
(none)

--- All detectors in intersection definition ---
✅ OK | ID= 1 | Name=VD11         | Type=NONE
✅ OK | ID= 2 | Name=VD25         | Type=NONE
✅ OK | ID= 3 | Name=VD41         | Type=NONE
✅ OK | ID= 4 | Name=VD42         | Type=NONE
✅ OK | ID= 5 | Name=D25          | Type=NONE
✅ OK | ID= 6 | Name=T1s          | Type=NONE
✅ OK | ID= 7 | Name=T2s          | Type=NONE
✅ OK | ID= 8 | Name=T3s          | Type=NONE
✅ OK | ID= 9 | Name=T4s          | Type=NONE
✅ OK | ID=10 | Name=T7s          | Type=NONE
✅ OK | ID=11 | Name=T8s          | Type=

**LD-LSA8 Failure Check**

In [4]:
alias_target = "LD-LSA8"

tlc, fp = load_tlc_by_alias(INTERSECTIONS_DIR, alias_target)
if tlc is None:
    raise FileNotFoundError(f"Could not find {alias_target} in {INTERSECTIONS_DIR}")

print_controller_summary(tlc, alias_target, fp)

all_detectors = extract_detectors(tlc)
fault_detectors = extract_fault_detectors(tlc)
fault_ids = {d_id for d_id, _, _ in fault_detectors}

print("\n--- Controller-reported faulty detectors (UNUSABLE) ---")
if not fault_detectors:
    print("(none)")
else:
    for d_id, d_name, d_type in fault_detectors:
        print(f"- ID={d_id} | Name={d_name} | Type={d_type}")

print("\n--- All detectors in intersection definition ---")
for d_id, d_name, d_type in all_detectors:
    tag = "❌ FAULT" if d_id in fault_ids else "✅ OK"
    print(f"{tag} | ID={d_id:>2} | Name={d_name:<12} | Type={d_type}")

print("\n=== USABLE (snapshot-level) ===")
for d_id, d_name, _ in all_detectors:
    if d_id not in fault_ids:
        print(f" ✅ ID={d_id} | {d_name}")

print("\n=== UNUSABLE (snapshot-level) ===")
if not fault_detectors:
    print("(none)")
else:
    for d_id, d_name, _ in fault_detectors:
        print(f" ❌ ID={d_id} | {d_name}")


STATE CHECK: LD-LSA8
File: C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\intersections\LSA8.json
name: LD-LSA8
alias: LD-LSA8
active: True
state: ERROR
brokenDetectorCount: 0
lastStatusUpdate: 2025-06-04T09:30:51
lastContacted: 2025-06-04T09:44:53.287
powerOk: True | emergencyOff: False | doorOpen: False | persistenceOk: False

--- Controller-reported faulty detectors (UNUSABLE) ---
(none)

--- All detectors in intersection definition ---
✅ OK | ID= 1 | Name=D2_1         | Type=NONE
✅ OK | ID= 2 | Name=D2_3         | Type=NONE
✅ OK | ID= 3 | Name=D4_1         | Type=NONE
✅ OK | ID= 4 | Name=D5_1         | Type=NONE
✅ OK | ID= 5 | Name=D6_1         | Type=NONE
✅ OK | ID= 6 | Name=T1s          | Type=NONE
✅ OK | ID= 7 | Name=T1b          | Type=NONE
✅ OK | ID= 8 | Name=T2s          | Type=NONE
✅ OK | ID= 9 | Name=T2b          | Type=NONE
✅ OK | ID=10 | Name=T3s          | Type=NONE
✅ OK | ID=11 | Name=T3b          | Type=NONE
✅ 

**LD-LSA1 Failure Check**

In [5]:
alias_target = "LD-LSA1"

tlc, fp = load_tlc_by_alias(INTERSECTIONS_DIR, alias_target)
if tlc is None:
    raise FileNotFoundError(f"Could not find {alias_target} in {INTERSECTIONS_DIR}")

print_controller_summary(tlc, alias_target, fp)

all_detectors = extract_detectors(tlc)
fault_detectors = extract_fault_detectors(tlc)
fault_ids = {d_id for d_id, _, _ in fault_detectors}

print("\n--- Controller-reported faulty detectors (UNUSABLE) ---")
if not fault_detectors:
    print("(none)")
else:
    for d_id, d_name, d_type in fault_detectors:
        print(f"- ID={d_id} | Name={d_name} | Type={d_type}")

print("\n--- All detectors in intersection definition ---")
for d_id, d_name, d_type in all_detectors:
    tag = "❌ FAULT" if d_id in fault_ids else "✅ OK"
    print(f"{tag} | ID={d_id:>2} | Name={d_name:<12} | Type={d_type}")

print("\n=== USABLE (snapshot-level) ===")
for d_id, d_name, _ in all_detectors:
    if d_id not in fault_ids:
        print(f" ✅ ID={d_id} | {d_name}")

print("\n=== UNUSABLE (snapshot-level) ===")
if not fault_detectors:
    print("(none)")
else:
    for d_id, d_name, _ in fault_detectors:
        print(f" ❌ ID={d_id} | {d_name}")


STATE CHECK: LD-LSA1
File: C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\intersections\LSA1.json
name: LD-LSA1
alias: LD-LSA1
active: True
state: INTERNAL_FAILURE
brokenDetectorCount: 2
lastStatusUpdate: 2025-06-04T09:30:00
lastContacted: 2025-06-04T09:44:00.761
powerOk: True | emergencyOff: False | doorOpen: False | persistenceOk: True

--- Controller-reported faulty detectors (UNUSABLE) ---
- ID=25 | Name=DET51 | Type=NONE
- ID=29 | Name=DET72 | Type=NONE

--- All detectors in intersection definition ---
✅ OK | ID= 7 | Name=DET11        | Type=NONE
✅ OK | ID= 8 | Name=DET12        | Type=NONE
✅ OK | ID= 9 | Name=DET21        | Type=NONE
✅ OK | ID=10 | Name=DET22        | Type=NONE
✅ OK | ID=11 | Name=DET31        | Type=NONE
✅ OK | ID=12 | Name=DET32        | Type=NONE
✅ OK | ID=13 | Name=DET41        | Type=NONE
✅ OK | ID=14 | Name=DET42        | Type=NONE
✅ OK | ID=15 | Name=TAS1         | Type=NONE
✅ OK | ID=16 | Name=TAS